# Rozhodovací stromy – celé téma v jednom cvičení

Rozhodovací strom je posloupnost otázek typu *„Je příjem nejvýše 32 tisíc?“*. Každá otázka rozdělí data na dvě části
a v každé části se ptáme dál, dokud nejsou skupiny „čisté“. Dnes projdeme celou cestu – od výpočtu na papíře až po modely,
které vyhrávají soutěže na tabulkových datech:

| část | obsah |
|---|---|
| 1 | **Nečistota a jedno dělení** – Gini, entropie, ruční volba prahu |
| 2 | **Nejlepší dělení a strom krok po kroku** – funkce vyzkouší všechny prahy, my dělíme dál |
| 3 | **Regresní strom** – tytéž funkce, jen rozptyl místo Gini a průměr v listu |
| 4 | **Obecný strom jako rekurze** – celý algoritmus v jedné funkci, srovnání se scikit-learn |
| 5 | **Reálná data** – metriky, trénovací / validační / testovací data, ruční ladění |
| 6 | **Automatické ladění** – křížová validace, grid search, prořezávání |
| 7 | **Ansámbly** – bagging, náhodný les, gradient boosting |

Funkce jsou **předpřipravené** – stačí je spouštět. Řádky, které máte měnit, jsou označené `# UPRAVIT`.
Úkoly označené ⭐ jsou volitelné – pokud nestíháte, přeskočte je (podrobněji je rozebírají samostatná cvičení 1–6 v této složce).

In [ ]:
from dataclasses import dataclass      # jednoduchá třída pro uzel stromu

import numpy as np                     # pole a matematické operace
import pandas as pd                    # tabulky
import matplotlib.pyplot as plt        # grafy
from IPython.display import display    # zobrazení tabulek

from sklearn import datasets, metrics
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text, plot_tree
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier, GradientBoostingRegressor,
                              HistGradientBoostingClassifier, AdaBoostClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RSTATE = 42                            # opakovatelné výsledky

# Část 1 – Nečistota a jedno dělení

## 1.1 Data

Banka chce podle **měsíčního příjmu** (v tisících Kč) a **věku** klienta odhadnout, zda **splatí úvěr** (1 = splatí, 0 = nesplatí).
Čísla v grafu jsou pořadová čísla řádků. Tabulku můžete kdykoli upravit a spustit notebook znovu.

In [ ]:
loans = pd.DataFrame(
    [   # příjem, věk, splatí
        (18, 25, 0), (22, 45, 0), (25, 60, 0), (30, 22, 0),
        (34, 50, 1), (38, 28, 0), (42, 55, 1), (45, 35, 1),
        (48, 24, 0), (55, 30, 1), (62, 48, 1), (70, 26, 1),
        # UPRAVIT: sem můžete přidat další řádky
    ],
    columns=["prijem", "vek", "splati"],
)
CLASS_NAMES = {0: "nesplatí", 1: "splatí"}
COLORS = {0: "tab:red", 1: "tab:green"}


def plot_loans(df, feature=None, threshold=None, ax=None, title=""):
    if ax is None:
        _, ax = plt.subplots(figsize=(5.5, 4))
    for cls, group in df.groupby("splati"):                                 # každá třída svou barvou
        ax.scatter(group["prijem"], group["vek"], s=80, c=COLORS[cls], edgecolor="k", label=CLASS_NAMES[cls])
    for i, row in df.iterrows():                                            # číslo řádku vedle bodu
        ax.annotate(str(i), (row["prijem"], row["vek"]), xytext=(5, 3), textcoords="offset points", fontsize=8)
    if feature == "prijem":
        ax.axvline(threshold, color="k", ls="--", label=f"prijem <= {threshold}")
    elif feature == "vek":
        ax.axhline(threshold, color="k", ls="--", label=f"vek <= {threshold}")
    ax.set(xlim=(loans["prijem"].min() - 5, loans["prijem"].max() + 5),   # stejné osy ve všech grafech
           ylim=(loans["vek"].min() - 5, loans["vek"].max() + 5),
           xlabel="prijem", ylabel="vek", title=title or f"{len(df)} vzorků")
    ax.legend(fontsize=8)
    return ax


display(loans.T)                                                            # tabulka „na ležato“
plot_loans(loans, title="Klienti banky")
plt.show()

**✏️ Úkol 1.1** – Kdybyste směli položit **jedinou** otázku tvaru *„příznak ≤ práh?“*, kterou byste zvolili? Zapište si ji.

## 1.2 Míry nečistoty

Uzel je **čistý**, když obsahuje vzorky jediné třídy. Nečistotu měříme pomocí podílů tříd $p_k$ v uzlu:

| míra | vzorec | čistý uzel | 2 třídy 50 : 50 |
|---|---|---|---|
| **Gini** | $G = 1 - \sum_k p_k^2$ | 0 | 0,5 |
| **Entropie** | $H = -\sum_k p_k \log_2 p_k$ | 0 | 1 |
| **Rozptyl** (regrese, část 3) | $\mathrm{Var} = \frac{1}{n}\sum_i (y_i - \bar y)^2$ | 0 | – |

Slovník `CRITERIA` ke každému kritériu přiřadí dvojici funkcí: **jak měřit nečistotu** a **co předpovídá list**
(nejčastější třídu, nebo průměr). Tohle je jediný rozdíl mezi klasifikačním a regresním stromem – vše ostatní je společné.

In [ ]:
def gini(y):
    p = np.unique(y, return_counts=True)[1] / len(y)     # podíly tříd v uzlu
    return 1 - np.sum(p ** 2)


def entropy(y):
    p = np.unique(y, return_counts=True)[1] / len(y)
    return np.sum(p * np.log2(1 / p))                    # totéž co -sum(p * log2(p))


def variance(y):
    return np.mean((y - np.mean(y)) ** 2)                # průměrná čtvercová odchylka od průměru


def majority(y):
    values, counts = np.unique(y, return_counts=True)
    return values[np.argmax(counts)]                     # nejčastější třída


CRITERIA = {                                             # kritérium -> (míra nečistoty, hodnota listu)
    "gini": (gini, majority),
    "entropy": (entropy, majority),
    "variance": (variance, np.mean),
}

for n0, n1 in [(12, 0), (9, 3), (6, 6)]:                 # UPRAVIT: složení uzlů (počet tříd 0 a 1)
    y_node = np.array([0] * n0 + [1] * n1)
    print(f"uzel ({n0:2d} x 0, {n1:2d} x 1):  Gini = {gini(y_node):.3f},  entropie = {entropy(y_node):.3f}")

## 1.3 Kvalita jednoho dělení

Dělení *„`příznak` ≤ `práh`?“* pošle vzorky s odpovědí **ano** do levého potomka, ostatní do pravého. Kvalitu posoudíme podle
**vážené nečistoty** potomků a **zisku** (poklesu nečistoty):

$$I_\text{dělení} = \frac{n_L}{n}\, I(L) + \frac{n_R}{n}\, I(R), \qquad \text{zisk} = I(\text{rodič}) - I_\text{dělení}$$

Hledáme dělení s **co největším ziskem**.

In [ ]:
def split_score(df, target, feature, threshold, criterion="gini"):
    impurity = CRITERIA[criterion][0]
    y = df[target].to_numpy()
    left = (df[feature] <= threshold).to_numpy()         # True = vzorek jde doleva
    n, n_l = len(y), left.sum()
    n_r = n - n_l
    imp_l = impurity(y[left]) if n_l else 0.0            # prázdný potomek nemá žádnou nečistotu
    imp_r = impurity(y[~left]) if n_r else 0.0
    weighted = (n_l * imp_l + n_r * imp_r) / n           # vážená nečistota potomků
    return {"příznak": feature, "práh": threshold, "n levý": n_l, "n pravý": n_r,
            "nečistota levý": imp_l, "nečistota pravý": imp_r,
            "vážená nečistota": weighted, "zisk": impurity(y) - weighted}


FEATURE, THRESHOLD = "prijem", 40                        # UPRAVIT: vaše dělení z úkolu 1.1

display(pd.DataFrame([split_score(loans, "splati", FEATURE, THRESHOLD)]).round(3))
plot_loans(loans, FEATURE, THRESHOLD, title=f"Dělení {FEATURE} <= {THRESHOLD}")
plt.show()

**✏️ Úkol 1.2**

* Spočítejte ručně Gini celé tabulky (kolik je vzorků třídy 0 a 1?) a ověřte: `gini(loans["splati"])`.
* Měňte `FEATURE` a `THRESHOLD` a hledejte co největší zisk. Jaký nejlepší zisk jste našli?

# Část 2 – Nejlepší dělení a strom krok po kroku

Prahů je nekonečně mnoho, ale **různých dělení** jen málo – stačí zkusit prahy **uprostřed mezi sousedními hodnotami**.
Funkce `all_splits` vyzkouší všechny prahy všech příznaků a seřadí je od nejlepšího. Parametr `min_samples_leaf` vyřadí dělení,
po kterých by některý potomek měl příliš málo vzorků (využijeme ho později).

In [ ]:
def all_splits(df, target, criterion="gini", min_samples_leaf=1):
    rows = []
    for feature in df.columns.drop(target):              # každý příznak
        values = np.unique(df[feature])                  # seřazené různé hodnoty
        for t in (values[:-1] + values[1:]) / 2:         # každý práh mezi sousedními hodnotami
            rows.append(split_score(df, target, feature, t, criterion))
    table = pd.DataFrame(rows)
    if table.empty:                                      # uzel nejde rozdělit (všechny vzorky stejné)
        return table
    table = table[(table["n levý"] >= min_samples_leaf) & (table["n pravý"] >= min_samples_leaf)]
    return table.sort_values("zisk", ascending=False, kind="stable").reset_index(drop=True)   # při shodě vyhraje dřívější


splits = all_splits(loans, "splati")
display(splits.head(5).round(3))
print(f"vyzkoušeno {len(splits)} dělení, nejlepší: {splits.loc[0, 'příznak']} <= {splits.loc[0, 'práh']}")

Teď strom stavíme **ručně krok po kroku**: rozdělíme kořen nejlepším dělením a pro každý potomek, který není čistý,
hledáme nejlepší dělení znovu – **jen z jeho vzorků**. List předpovídá nejčastější třídu svých vzorků.

In [ ]:
def describe(df, name):
    y = df["splati"]
    print(f"uzel {name:3s}: n = {len(y):2d}, třídy {y.value_counts().sort_index().to_dict()}, "
          f"Gini = {gini(y):.3f}" + ("  -> čistý list" if gini(y) == 0 else "  -> dělíme dál"))


def split(df, feature, threshold):
    return df[df[feature] <= threshold], df[df[feature] > threshold]


nodes = {"kořen": loans}                                 # všechny dosud vytvořené uzly podle jména


def split_node(name):
    df = nodes[name]
    best = all_splits(df, "splati").loc[0]               # nejlepší dělení – jen ze vzorků tohoto uzlu
    left, right = split(df, best["příznak"], best["práh"])
    prefix = "" if name == "kořen" else name
    nodes[prefix + "L"], nodes[prefix + "P"] = left, right   # potomci dostanou jména např. P -> PL, PP
    print(f"uzel {name}: {best['příznak']} <= {best['práh']}")
    describe(left, prefix + "L")
    describe(right, prefix + "P")
    plot_loans(df, best["příznak"], best["práh"], title=f"Uzel {name} a jeho nejlepší dělení")
    plt.show()


split_node("kořen")                                      # 1. krok: kořen -> L, P
split_node("P")                                          # 2. krok: uzel P -> PL, PP

**✏️ Úkol 2.1**

* Shoduje se nejlepší dělení kořene s vaším odhadem z úkolu 1.1?
* Uzel `PL` ještě není čistý. V buňce níže ho rozdělte a pokračujte, dokud nejsou všechny listy čisté. Nakreslete si strom na papír.
* ⭐ Změňte kritérium na `"entropy"` (`all_splits(loans, "splati", "entropy")`). Změní se pořadí dělení?

In [ ]:
split_node("PL")                                         # UPRAVIT: další uzel, který není čistý

Opakujeme pořád totéž: *najdi nejlepší dělení → rozděl → pokračuj v potomcích*. To je **rekurze** – v části 4 ji napíšeme jako funkci.

# Část 3 – Regresní strom

U regrese je cílem **číslo**. Postup je stejný, mění se jen dvě věci:

* nečistotou je **rozptyl** – dělení je dobré, když jsou ceny v obou potomcích podobné;
* list předpovídá **průměr** svých vzorků (průměr minimalizuje součet čtverců odchylek).

Data: **stáří** ojetého auta (roky) a jeho **cena** (tisíce Kč).

In [ ]:
cars = pd.DataFrame(
    [(1, 520), (2, 480), (3, 450), (4, 300), (5, 280),
     (6, 310), (8, 150), (9, 170), (11, 90), (12, 60)],   # UPRAVIT: stáří, cena
    columns=["stari", "cena"],
)

splits = all_splits(cars, "cena", criterion="variance")  # stejná funkce, jiné kritérium
display(splits.head(3).round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
by_t = splits.sort_values("práh")
axes[0].plot(by_t["práh"], by_t["zisk"], "o-")
axes[0].set(xlabel="práh pro stari", ylabel="zisk (pokles rozptylu)", title="Zisk každého prahu")
t = splits.loc[0, "práh"]
left, right = split(cars, "stari", t)
axes[1].scatter(cars["stari"], cars["cena"], s=60, edgecolor="k")
axes[1].hlines([left["cena"].mean(), right["cena"].mean()], [0, t], [t, cars["stari"].max() + 1], colors="tab:red", lw=2)
axes[1].axvline(t, color="k", ls="--")
axes[1].set(xlabel="stari", ylabel="cena", title=f"Nejlepší dělení stari <= {t}: předpověď = průměr listu")
for ax in axes:
    ax.grid(alpha=0.3)
plt.show()

**✏️ Úkol 3.1**

* Jakou cenu předpoví strom s jedním dělením pro auto staré 7 let? A pro auto staré 30 let? (Strom **neumí extrapolovat**.)
* ⭐ Změňte cenu prvního auta na `2000` (odlehlá hodnota). Jak se změní nejlepší dělení? Proč rozptyl „obětuje“ celé první dělení jedinému autu?

# Část 4 – Obecný strom jako rekurze

Celý algoritmus se vejde do jedné funkce:

```
postav_strom(data):
    vytvoř uzel s předpovědí (nejčastější třída / průměr)
    pokud je uzel čistý nebo je dosažena max. hloubka:  vrať list
    najdi nejlepší dělení (all_splits)
    pokud žádné dělení nesníží nečistotu:              vrať list
    uzel.levý  = postav_strom(data, kde příznak <= práh)
    uzel.pravý = postav_strom(data, kde příznak >  práh)
```

Funkce `build_tree` používá **tytéž** funkce `all_splits` a `CRITERIA` jako ruční výpočty výše – nic nového se v ní neskrývá.

In [ ]:
@dataclass
class Node:
    value: object                  # předpověď uzlu (třída nebo průměr)
    n_samples: int                 # počet trénovacích vzorků
    impurity: float                # nečistota uzlu
    feature: str = None            # podle čeho dělíme (None = list)
    threshold: float = None        # vlevo jdou vzorky s hodnotou <= threshold
    left: "Node" = None
    right: "Node" = None


TRACE = False                                            # UPRAVIT: True = vypisovat průběh stavby


def build_tree(df, target, criterion="gini", max_depth=None, min_samples_leaf=1, depth=0):
    impurity, leaf_value = CRITERIA[criterion]
    y = df[target].to_numpy()
    node = Node(value=leaf_value(y), n_samples=len(y), impurity=impurity(y))
    if node.impurity == 0 or (max_depth is not None and depth >= max_depth):
        return node                                      # čistý uzel nebo maximální hloubka -> list
    splits = all_splits(df, target, criterion, min_samples_leaf)
    if splits.empty or splits.loc[0, "zisk"] <= 0:
        return node                                      # žádné užitečné dělení -> list
    node.feature, node.threshold = splits.loc[0, "příznak"], splits.loc[0, "práh"]
    if TRACE:
        print("    " * depth + f"{node.feature} <= {node.threshold}  (n = {node.n_samples})")
    mask = df[node.feature] <= node.threshold
    node.left = build_tree(df[mask], target, criterion, max_depth, min_samples_leaf, depth + 1)     # rekurze
    node.right = build_tree(df[~mask], target, criterion, max_depth, min_samples_leaf, depth + 1)
    return node


def predict(node, df):
    out = []
    for _, row in df.iterrows():
        n = node
        while n.feature is not None:                     # dokud nejsme v listu, jdeme doleva / doprava
            n = n.left if row[n.feature] <= n.threshold else n.right
        out.append(n.value)
    return np.array(out)


def print_tree(node, indent=""):
    if node.feature is None:
        print(f"{indent}-> {node.value:.4g}   (n = {node.n_samples})")
        return
    print(f"{indent}{node.feature} <= {node.threshold:.4g}")
    print_tree(node.left, indent + "|   ")
    print(f"{indent}{node.feature} >  {node.threshold:.4g}")
    print_tree(node.right, indent + "|   ")

Kontrola: náš strom a strom ze scikit-learn na datech z části 1 a 3.

In [ ]:
root = build_tree(loans, "splati", criterion="gini")
print("=== náš klasifikační strom ===")
print_tree(root)
sk = DecisionTreeClassifier(random_state=0).fit(loans[["prijem", "vek"]], loans["splati"])
print("=== scikit-learn ===")
print(export_text(sk, feature_names=["prijem", "vek"]))

car_tree = build_tree(cars, "cena", criterion="variance", max_depth=2)
print("=== náš regresní strom (max_depth=2) ===")
print_tree(car_tree)
sk_reg = DecisionTreeRegressor(max_depth=2).fit(cars[["stari"]], cars["cena"])
print("stejné předpovědi jako scikit-learn:", np.allclose(predict(car_tree, cars), sk_reg.predict(cars[["stari"]])))

**Pravidla pro zastavení.** Strom bez omezení roste, dokud nejsou všechny listy čisté – na trénovacích datech je pak bezchybný,
ale „naučí se nazpaměť“ i šum. Proto růst omezujeme hyperparametry `max_depth` (maximální hloubka) nebo `min_samples_leaf`
(nejmenší počet vzorků v listu). Graf ukazuje regresní strom pro různé hloubky:

In [ ]:
grid_x = pd.DataFrame({"stari": np.linspace(0, 13, 300)})
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)
for ax, depth in zip(axes, [1, 2, 3, None]):             # UPRAVIT: hloubky
    tree = build_tree(cars, "cena", "variance", max_depth=depth)    # UPRAVIT: zkuste min_samples_leaf=2
    ax.scatter(cars["stari"], cars["cena"], s=50, edgecolor="k")
    ax.plot(grid_x["stari"], predict(tree, grid_x), "tab:red")
    ax.set(title=f"max_depth = {depth}", xlabel="stari")
axes[0].set_ylabel("cena")
plt.show()

**✏️ Úkol 4.1**

* Nastavte `TRACE = True`, spusťte buňku s `Node` a pak kontrolní buňku znovu. V jakém pořadí rekurze uzly prochází?
* Který ze čtyř regresních stromů byste zvolili pro předpověď ceny dalšího auta? Proč ne ten nejhlubší?
* ⭐ **XOR**: spusťte buňku níže. Proč náš strom data vůbec nerozdělí, i když dva řezy by stačily? Co to říká o **hladovém** hledání dělení?
  Scikit-learn dělí i při nulovém zisku, a tak XOR „náhodou“ vyřeší – stačí ale přidat šum a první dělení bude dál vybrané náhodně.

In [ ]:
xor = pd.DataFrame({"x1": [0, 0, 1, 1], "x2": [0, 1, 0, 1], "y": [0, 1, 1, 0]})
print_tree(build_tree(xor, "y"))                         # náš strom: zastaví, když žádné dělení nemá kladný zisk
display(all_splits(xor, "y")[["příznak", "práh", "zisk"]])
sk_xor = DecisionTreeClassifier(random_state=0).fit(xor[["x1", "x2"]], xor["y"])
print(export_text(sk_xor, feature_names=["x1", "x2"]))     # scikit-learn: dělí i při nulovém zisku

# Část 5 – Reálná data: metriky a ruční ladění

Teď použijeme strom ze scikit-learn (dělá totéž co `build_tree`, jen rychleji) na datech **Breast Cancer Wisconsin**:
569 nádorů popsaných 30 příznaky, cílem je poznat **zhoubný** nádor (1).

Data rozdělíme na tři části:

| část | podíl | k čemu slouží |
|---|---|---|
| **trénovací** | 60 % | učení stromu |
| **validační** | 20 % | porovnání nastavení – podle ní volíme **hyperparametry** (`max_depth`, `min_samples_leaf`, …) |
| **testovací** | 20 % | **jediné** závěrečné vyhodnocení – na konci cvičení |

**Metriky** (pozitivní třída = zhoubný):

* **správnost** (accuracy) – podíl správných předpovědí;
* **přesnost** (precision) – z nádorů označených jako zhoubné, kolik jich opravdu zhoubných je;
* **úplnost** (recall) – kolik zhoubných nádorů model našel (u diagnózy nejdůležitější: přehlédnutý nádor je drahá chyba);
* **F1** – harmonický průměr přesnosti a úplnosti; tu budeme optimalizovat.

In [ ]:
cancer = datasets.load_breast_cancer(as_frame=True)
X = cancer.data
y = (cancer.target == 0).astype(int)                     # 1 = zhoubný (pozitivní třída)

X_trval, X_test, y_trval, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RSTATE)
X_train, X_val, y_train, y_val = train_test_split(X_trval, y_trval, test_size=0.25, stratify=y_trval,
                                                  random_state=RSTATE)        # 0,25 * 80 % = 20 %
print(f"trénovací: {len(y_train)}, validační: {len(y_val)}, testovací: {len(y_test)} vzorků, "
      f"podíl zhoubných: {y.mean():.2f}")


def scores(y_true, y_pred):
    return {"správnost": metrics.accuracy_score(y_true, y_pred),
            "přesnost": metrics.precision_score(y_true, y_pred, zero_division=0),
            "úplnost": metrics.recall_score(y_true, y_pred),
            "F1": metrics.f1_score(y_true, y_pred)}


baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)   # vždy „nezhoubný“
full_tree = DecisionTreeClassifier(random_state=RSTATE).fit(X_train, y_train)  # strom bez omezení
display(pd.DataFrame({
    "baseline (validační)": scores(y_val, baseline.predict(X_val)),
    "strom (trénovací)": scores(y_train, full_tree.predict(X_train)),
    "strom (validační)": scores(y_val, full_tree.predict(X_val)),
}).T.round(3))
print(f"strom bez omezení: hloubka {full_tree.get_depth()}, listů {full_tree.get_n_leaves()}")
metrics.ConfusionMatrixDisplay.from_predictions(y_val, full_tree.predict(X_val),
                                                display_labels=["nezhoubný", "zhoubný"], cmap="Blues")
plt.title("Matice záměn – validační data")
plt.show()

Baseline má slušnou správnost, ale nulovou úplnost – proto u nevyvážených tříd nestačí sledovat správnost.
Strom bez omezení je na trénovacích datech bezchybný a na validačních horší: **přeučení**.

## 5.1 Ruční ladění hyperparametrů

Do seznamu `ATTEMPTS` přidávejte nastavení a sledujte **validační F1**. Tabulka je seřazená od nejlepšího pokusu.

| hyperparametr | co dělá |
|---|---|
| `max_depth` | maximální hloubka stromu |
| `min_samples_leaf` | nejmenší počet vzorků v listu |
| `min_samples_split` | nejmenší počet vzorků v uzlu, aby se smel dělit |
| `max_leaf_nodes` | maximální počet listů |
| `criterion` | `"gini"` nebo `"entropy"` |
| `class_weight` | `"balanced"` = chyby na menší třídě váží víc |

In [ ]:
ATTEMPTS = [                                             # UPRAVIT: přidávejte další pokusy
    {},
    {"max_depth": 3},
    {"max_depth": 4, "min_samples_leaf": 5},
]

log = []
for params in ATTEMPTS:
    tree = DecisionTreeClassifier(random_state=RSTATE, **params).fit(X_train, y_train)
    row = {"hyperparametry": str(params), "hloubka": tree.get_depth(), "listů": tree.get_n_leaves(),
           "trén. F1": metrics.f1_score(y_train, tree.predict(X_train))}
    row.update({f"val. {k}": v for k, v in scores(y_val, tree.predict(X_val)).items()})
    log.append(row)
display(pd.DataFrame(log).sort_values("val. F1", ascending=False).round(3))

Graf ukazuje vliv jednoho hyperparametru: trénovací F1 roste se složitostí stromu, validační F1 od jistého bodu klesá.

In [ ]:
PARAM, VALUES = "max_depth", [1, 2, 3, 4, 5, 6, 8, 10, None]   # UPRAVIT: např. "min_samples_leaf", [1, 2, 5, 10, 20, 40]

train_f1, val_f1 = [], []
for v in VALUES:
    tree = DecisionTreeClassifier(random_state=RSTATE, **{PARAM: v}).fit(X_train, y_train)
    train_f1.append(metrics.f1_score(y_train, tree.predict(X_train)))
    val_f1.append(metrics.f1_score(y_val, tree.predict(X_val)))
plt.figure(figsize=(7, 3.5))
plt.plot([str(v) for v in VALUES], train_f1, "o-", label="trénovací")
plt.plot([str(v) for v in VALUES], val_f1, "s-", label="validační")
plt.xlabel(PARAM); plt.ylabel("F1"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

**✏️ Úkol 5.1** – Najděte ručně nastavení s co nejlepší validační F1 (aspoň 5 pokusů). Své nejlepší nastavení zapište do `MANUAL`:

In [ ]:
MANUAL = {"max_depth": 4, "min_samples_leaf": 5}         # UPRAVIT: vaše nejlepší nastavení

tree = DecisionTreeClassifier(random_state=RSTATE, **MANUAL).fit(X_train, y_train)
importance = pd.Series(tree.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Nejdůležitější příznaky (pokles nečistoty, který dělení podle příznaku přinesla):")
display(importance[importance > 0].round(3).to_frame("důležitost"))

plt.figure(figsize=(14, 5))
plot_tree(tree, max_depth=2, feature_names=list(X.columns), class_names=["nezhoubný", "zhoubný"],
          filled=True, rounded=True, fontsize=8)
plt.title("Horní část stromu")
plt.show()

**✏️ Úkol 5.2** – ⭐ Změňte `random_state` při dělení dat (`RSTATE` v první buňce) a spusťte část 5 znovu.
Zůstane nejlepší nastavení i nejdůležitější příznak stejný? Stromy jsou **nestabilní**: malá změna dat může změnit celý strom.
Jedna validační množina (114 vzorků) je navíc dost malá – volba podle ní je trochu náhodná. Obojí řeší další dvě části.

# Část 6 – Automatické ladění

## 6.1 Křížová validace

Místo jedné validační množiny rozdělíme trénovací + validační data (80 %) na $k$ částí (**foldů**). Model naučíme $k$-krát,
pokaždé na $k-1$ částech, a vyhodnotíme na zbývající. Výsledkem je průměr **a** rozptyl odhadu – každý vzorek je jednou validační.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RSTATE)    # stratifikace = stejné podíly tříd ve foldech
f1 = cross_val_score(DecisionTreeClassifier(random_state=RSTATE, **MANUAL), X_trval, y_trval, cv=cv, scoring="f1")
print("F1 v jednotlivých foldech:", f1.round(3))
print(f"průměr {f1.mean():.3f} ± {f1.std():.3f}")

## 6.2 Grid search

`GridSearchCV` vyzkouší **všechny kombinace** hodnot z mřížky, každou ohodnotí křížovou validací a vybere nejlepší.
Je to přesně náš ruční postup z části 5 – jen systematicky a s lepším odhadem kvality.

In [ ]:
param_grid = {                                           # UPRAVIT: mřížka hodnot
    "max_depth": [2, 3, 4, 5, 6, 8, None],
    "min_samples_leaf": [1, 2, 5, 10, 20],
    "criterion": ["gini", "entropy"],
}
grid = GridSearchCV(DecisionTreeClassifier(random_state=RSTATE), param_grid, cv=cv, scoring="f1")
grid.fit(X_trval, y_trval)
n = len(grid.cv_results_["params"])
print(f"{n} kombinací x 5 foldů = {5 * n} naučených stromů")
print("nejlepší hyperparametry:", grid.best_params_)
print(f"nejlepší F1 v křížové validaci: {grid.best_score_:.3f}")

res = pd.DataFrame(grid.cv_results_)
res = res[res["param_criterion"] == grid.best_params_["criterion"]]
heat = res.pivot_table(index=res["params"].apply(lambda p: str(p["max_depth"])),
                       columns="param_min_samples_leaf", values="mean_test_score", sort=False)
fig, ax = plt.subplots(figsize=(6.5, 4.5))
im = ax.imshow(heat.values, cmap="viridis")
ax.set(xticks=range(heat.shape[1]), xticklabels=heat.columns, yticks=range(heat.shape[0]), yticklabels=heat.index,
       xlabel="min_samples_leaf", ylabel="max_depth", title=f"CV F1 (criterion = {grid.best_params_['criterion']})")
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        ax.text(j, i, f"{heat.values[i, j]:.3f}", ha="center", va="center", color="w", fontsize=8)
plt.colorbar(im, ax=ax)
plt.show()

Při velkých mřížkách je počet kombinací obrovský – pak se používá `RandomizedSearchCV`, který vyzkouší jen zadaný počet
**náhodných** kombinací (často stejně dobře za zlomek času).

## 6.3 Prořezávání (cost complexity pruning)

Jiný přístup: necháme strom vyrůst celý a pak ho **ořežeme**. Parametr `ccp_alpha` je „cena za list“ – strom minimalizuje
$\text{chyba} + \alpha \cdot \text{počet listů}$. Větší $\alpha$ = menší strom. Scikit-learn spočítá všechny hodnoty $\alpha$,
při kterých se strom zmenší; nejlepší vybereme křížovou validací.

In [ ]:
path = DecisionTreeClassifier(random_state=RSTATE).cost_complexity_pruning_path(X_trval, y_trval)
rows = []
for alpha in path.ccp_alphas[:-1]:                       # poslední alfa ořízne strom až na kořen
    t = DecisionTreeClassifier(ccp_alpha=alpha, random_state=RSTATE)
    rows.append({"ccp_alpha": alpha, "listů": t.fit(X_trval, y_trval).get_n_leaves(),
                 "CV F1": cross_val_score(t, X_trval, y_trval, cv=cv, scoring="f1").mean()})
pruning = pd.DataFrame(rows)
best_alpha = float(pruning.loc[pruning["CV F1"] == pruning["CV F1"].max(), "ccp_alpha"].iloc[-1])   # při shodě menší strom

plt.figure(figsize=(7, 3.5))
plt.plot(pruning["listů"], pruning["CV F1"], "o-")
plt.xlabel("počet listů prořezaného stromu"); plt.ylabel("CV F1"); plt.grid(alpha=0.3)
plt.title(f"Prořezávání: nejlepší ccp_alpha = {best_alpha:.4f}")
plt.show()

## 6.4 Srovnání na testovacích datech

Teprve teď – po výběru hyperparametrů – použijeme testovací data. Každý model naučíme na trénovacích + validačních datech
a **jednou** vyhodnotíme.

In [ ]:
candidates = {
    "strom bez omezení": {},
    "ruční ladění (část 5)": MANUAL,
    "grid search": grid.best_params_,
    "prořezávání": {"ccp_alpha": best_alpha},
}
rows = []
for name, params in candidates.items():
    t = DecisionTreeClassifier(random_state=RSTATE, **params).fit(X_trval, y_trval)
    rows.append({"způsob": name, "listů": t.get_n_leaves(),
                 "CV F1": cross_val_score(t, X_trval, y_trval, cv=cv, scoring="f1").mean(),
                 **{f"test {k}": v for k, v in scores(y_test, t.predict(X_test)).items()}})
display(pd.DataFrame(rows).round(3))

**✏️ Úkol 6.1**

* Vyhrál na testu ten model, který měl nejlepší CV F1? Testovací množina má jen 114 vzorků – jeden nádor navíc nebo méně
  posune F1 zhruba o 0,02. Rozdíly menší než to nejsou průkazné.
* Proč nesmíme hyperparametry vybírat podle testovacích dat?
* ⭐ Přidejte do mřížky `"class_weight": [None, "balanced"]`. Zlepší se úplnost?

# Část 7 – Ansámbly: mnoho stromů je lepší než jeden

Jeden hluboký strom má malé **zkreslení** (umí popsat složitý vztah), ale velký **rozptyl** (každá změna dat ho výrazně změní).
Ansámbly kombinují mnoho stromů:

| metoda | myšlenka | co zlepšuje |
|---|---|---|
| **bagging** | mnoho hlubokých stromů na náhodných výběrech dat, výsledky **zprůměrujeme** | rozptyl |
| **náhodný les** | bagging + v každém dělení jen náhodná podmnožina příznaků (stromy jsou si méně podobné) | rozptyl |
| **boosting** | malé stromy **postupně**, každý opravuje chyby předchozích | zkreslení |

## 7.1 Bagging ručně

**Bootstrap** = výběr $n$ vzorků z $n$ **s vracením** (některé se opakují, asi třetina chybí). Na každém výběru naučíme hluboký strom
a předpovědi zprůměrujeme. Data: zašuměný sinus.

In [ ]:
rng = np.random.default_rng(0)
def make_sine(n):
    x = rng.uniform(0, 6, size=n)
    return x.reshape(-1, 1), np.sin(x) + rng.normal(0, 0.3, size=n)   # sinus + šum

X_tr, y_tr = make_sine(80)
X_te, y_te = make_sine(500)
xs = np.linspace(0, 6, 500).reshape(-1, 1)


def bootstrap(X, y, rng):
    idx = rng.integers(0, len(y), size=len(y))           # náhodné indexy s vracením
    return X[idx], y[idx]


def bagging_predict(X_train, y_train, X_new, n_trees, seed=0):
    rng = np.random.default_rng(seed)
    predictions = []
    for _ in range(n_trees):
        Xb, yb = bootstrap(X_train, y_train, rng)
        predictions.append(DecisionTreeRegressor().fit(Xb, yb).predict(X_new))
    return np.mean(predictions, axis=0)                  # průměr přes všechny stromy


fig, axes = plt.subplots(1, 3, figsize=(15, 3.6), sharey=True)
for ax, n_trees in zip(axes, [1, 10, 200]):              # UPRAVIT: počty stromů
    mse = np.mean((bagging_predict(X_tr, y_tr, X_te, n_trees) - y_te) ** 2)
    ax.scatter(X_tr, y_tr, s=12)
    ax.plot(xs, np.sin(xs), "k--", lw=1)
    ax.plot(xs, bagging_predict(X_tr, y_tr, xs, n_trees), "tab:red")
    ax.set_title(f"{n_trees} stromů, test MSE {mse:.3f}  (šum: {0.3 ** 2:.2f})")
plt.show()

## 7.2 Gradient boosting ručně

Boosting začne konstantní předpovědí (průměrem) a pak opakuje: spočítej **zbytky** (co model zatím nevysvětlil), nauč na nich
**malý** strom a přičti jeho předpověď vynásobenou **rychlostí učení** (learning rate). Celé je to pár řádků:

In [ ]:
def gradient_boosting(X, y, n_trees=100, learning_rate=0.1, max_depth=2):
    f0 = y.mean()                                        # začátek: konstantní předpověď
    pred = np.full(len(y), f0)
    trees = []
    for _ in range(n_trees):
        residuals = y - pred                             # co zatím model nevysvětlil
        tree = DecisionTreeRegressor(max_depth=max_depth, random_state=0).fit(X, residuals)
        pred += learning_rate * tree.predict(X)          # malý krok směrem ke zbytkům
        trees.append(tree)
    return f0, trees


def boosting_predict(f0, trees, X, learning_rate=0.1):
    return f0 + learning_rate * sum(t.predict(X) for t in trees)


LR = 0.1                                                 # UPRAVIT: rychlost učení
f0, trees = gradient_boosting(X_tr, y_tr, n_trees=300, learning_rate=LR)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)
for ax, m in zip(axes, [1, 10, 50, 300]):                # UPRAVIT: zobrazené počty stromů
    mse = np.mean((boosting_predict(f0, trees[:m], X_te, LR) - y_te) ** 2)
    ax.scatter(X_tr, y_tr, s=12)
    ax.plot(xs, np.sin(xs), "k--", lw=1)
    ax.plot(xs, boosting_predict(f0, trees[:m], xs, LR), "tab:red")
    ax.set_title(f"po {m} stromech, test MSE {mse:.3f}")
plt.show()

sk_gb = GradientBoostingRegressor(n_estimators=300, learning_rate=LR, max_depth=2, random_state=0).fit(X_tr, y_tr)
print(f"největší rozdíl oproti scikit-learn: {np.abs(boosting_predict(f0, trees, X_te, LR) - sk_gb.predict(X_te)).max():.1e}")

**✏️ Úkol 7.1**

* Proč bagging používá **hluboké** stromy a boosting **mělké**?
* Nastavte `LR = 1.0` a `LR = 0.01`. Co se stane po 10 a po 300 stromech? (Menší rychlost učení = potřeba více stromů,
  ale obvykle lepší výsledek. Příliš mnoho stromů vede u boostingu k přeučení.)

## 7.3 Srovnání na reálných datech

Zpět k nádorům: všechny modely porovnáme křížovou validací na stejných datech jako v části 6. Pro srovnání přidáme
i jednoduchou **logistickou regresi** – stromy nejsou vždy nejlepší volbou.

In [ ]:
models = {                                               # UPRAVIT: přidejte / změňte modely
    "strom bez omezení": DecisionTreeClassifier(random_state=RSTATE),
    "naladěný strom (grid search)": DecisionTreeClassifier(random_state=RSTATE, **grid.best_params_),
    "náhodný les (300 stromů)": RandomForestClassifier(n_estimators=300, random_state=RSTATE),
    "AdaBoost": AdaBoostClassifier(n_estimators=200, random_state=RSTATE),
    "gradient boosting": GradientBoostingClassifier(random_state=RSTATE),
    "hist. gradient boosting": HistGradientBoostingClassifier(random_state=RSTATE),
    "logistická regrese": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
}
rows = []
for name, model in models.items():
    f1 = cross_val_score(model, X_trval, y_trval, cv=cv, scoring="f1")
    rows.append({"model": name, "CV F1": f1.mean(), "směr. odchylka": f1.std()})
display(pd.DataFrame(rows).sort_values("CV F1", ascending=False).round(3))

forest = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=RSTATE).fit(X_trval, y_trval)
print(f"náhodný les – OOB správnost (odhad bez validačních dat): {forest.oob_score_:.3f}")

**OOB (out-of-bag)**: každý strom lesa neviděl asi třetinu vzorků (nebyly v jeho bootstrapu). Na nich ho můžeme vyhodnotit –
dostaneme odhad kvality „zadarmo“, bez křížové validace.

**Další možnosti**, se kterými se v praxi setkáte:

* **XGBoost, LightGBM, CatBoost** – rychlé a vyladěné knihovny pro gradient boosting (regularizace, chybějící hodnoty,
  kategorické příznaky); na tabulkových datech patří k nejsilnějším modelům. `HistGradientBoostingClassifier` je jejich obdoba ve scikit-learn.
* **Extra Trees** – náhodný les, který navíc volí prahy náhodně.
* **Voting / stacking** – kombinace různých modelů (např. lesa a logistické regrese) průměrem nebo dalším modelem.

## 7.4 Závěrečný test

**✏️ Úkol 7.2** – Podle CV F1 vyberte jeden model, naučte ho na trénovacích + validačních datech a **jednou** ho vyhodnoťte na testu.

In [ ]:
FINAL = RandomForestClassifier(n_estimators=300, random_state=RSTATE)   # UPRAVIT: váš vybraný model

FINAL.fit(X_trval, y_trval)
pred = FINAL.predict(X_test)
display(pd.DataFrame([scores(y_test, pred)]).round(3))
metrics.ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["nezhoubný", "zhoubný"], cmap="Blues")
plt.show()

# Shrnutí

* Strom opakovaně dělí data otázkami *„příznak ≤ práh?“*; v každém uzlu volí dělení s největším **poklesem nečistoty** (hladově).
* **Klasifikace**: Gini / entropie, list = nejčastější třída. **Regrese**: rozptyl, list = průměr. Jinak je algoritmus stejný.
* Strom bez omezení se **přeučí**. Růst omezujeme (`max_depth`, `min_samples_leaf`, …) nebo strom **prořežeme** (`ccp_alpha`).
* Hyperparametry volíme na validačních datech nebo **křížovou validací** (`GridSearchCV`); test používáme **jednou**, na konci.
* Jeden strom je čitelný, ale nestabilní. **Náhodný les** (průměr mnoha stromů) snižuje rozptyl, **boosting** (postupné opravy) zkreslení.
* Vždy srovnejte s baseline a s jednoduchým modelem – někdy vyhraje logistická regrese.

# Úkoly na doma (⭐)

1. **Regrese na reálných datech** – zopakujte části 5 a 6 pro `datasets.load_diabetes()` s `DecisionTreeRegressor`
   a metrikami MAE, RMSE a $R^2$. Porovnejte s `LinearRegression`. (Podrobně: cvičení 4 a 5.)
2. **Nové kritérium** – přidejte do `CRITERIA` kritérium `"mae"`: nečistota = průměrná absolutní odchylka od mediánu, list = medián.
   Postavte s ním strom na `cars` s odlehlou hodnotou z úkolu 3.1. (Podrobně: cvičení 2 a 3.)
3. **Ladění lesa** – pomocí `GridSearchCV` nalaďte u `RandomForestClassifier` parametry `max_features` a `min_samples_leaf`. (Podrobně: cvičení 6.)